In [ ]:
import tarfile
import gzip
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy
import warnings
from numpy.polynomial import Polynomial
from scipy.signal import savgol_filter, find_peaks
from scipy.optimize import curve_fit
from scipy.optimize import minimize_scalar
import requests
from pathlib import Path
from IPython.display import display

In [ ]:
Hinkle = tarfile.open("J_PASP_107_1042.tar.gz", "r:gz")
Hinkle.extractall("espectros", filter="data")

Nave = tarfile.open("J_ApJS_94_221.tar.gz", "r:gz")
Nave.extractall("lineas", filter="data")

In [ ]:
archivos_de_espectro = [
    name for name in Hinkle.getnames()
    if "ab" in name and name.endswith(".gz")
]

print(len(archivos_de_espectro))

In [ ]:
todos_verano = []
todos_invierno = []

for file_name in archivos_de_espectro:
    file = Hinkle.extractfile(file_name)
    with gzip.open(file, 'rt') as f:
        df = pd.read_csv(
            f,
            sep=r'\s+',
            header=None
        )


    col1 = pd.to_numeric(df[0], errors='coerce')
    col2 = pd.to_numeric(df[1], errors='coerce')
    col3 = pd.to_numeric(df[2], errors='coerce')
    col4 = pd.to_numeric(df[3], errors='coerce')
    col5 = pd.to_numeric(df[4], errors='coerce')
    col6 = pd.to_numeric(df[5], errors='coerce')
    col7 = pd.to_numeric(df[6], errors='coerce')


    temp = pd.DataFrame({
        "wavenumber": col1,
        "intensity": col4
    })

    temp1 = pd.DataFrame({
        "wavenumber": col1,
        "intensity": col7
    })

    todos_verano.append(temp)
    todos_invierno.append(temp1)

Sabemos que $$\lambda =
\frac{1}{\tilde{\nu}}$$

Y $$1 cm = 10^{8} Å$$

In [ ]:
espectro_verano = pd.concat(todos_verano)
espectro_invierno = pd.concat(todos_invierno)

espectro_verano["lambda"] = 1e8 / espectro_verano["wavenumber"]
espectro_invierno["lambda"] = 1e8 / espectro_invierno["wavenumber"]

espectro_verano.loc[(espectro_verano["intensity"] < 0) | (espectro_verano["intensity"] > 1.08), "intensity"] = np.nan
espectro_invierno.loc[(espectro_invierno["intensity"] < 0) | (espectro_invierno["intensity"] > 1.08), "intensity"] = np.nan

espectro_verano = espectro_verano.sort_values("lambda")
espectro_invierno = espectro_invierno.sort_values("lambda")

for lam in (espectro_verano, espectro_invierno):
    lam.loc[(lam["intensity"] < 0.05) | (lam["intensity"] > 1.08), "intensity"] = np.nan

verano = espectro_verano.drop_duplicates(
    subset="wavenumber"
)

invierno = espectro_invierno.drop_duplicates(
    subset="wavenumber"
)

verano = verano.reset_index(drop=True)
invierno = invierno.reset_index(drop=True)

print("Espectro verano:")
display(verano)

print("\nEspectro invierno:")
display(invierno)

In [ ]:
lambdai = 10000
lambdaf = 10080

region_verano = verano[
    (verano["lambda"] > lambdai) &
    (verano["lambda"] < lambdaf)
]



plt.figure(figsize=(10,4))

plt.plot(
    region_verano["lambda"],
    region_verano["intensity"],
    color="red"
)

plt.xlabel("Longitud de onda (Å)")
plt.ylabel("Intensidad normalizada")


plt.title(f"Región {lambdai} - {lambdaf} Å Verano")

plt.show()

In [ ]:
region_invierno = invierno[
    (invierno["lambda"] > lambdai) &
    (invierno["lambda"] < lambdaf)
]

plt.figure(figsize=(10,4))

plt.plot(
    region_invierno["lambda"],
    region_invierno["intensity"],
    color="blue"
)

plt.xlabel("Longitud de onda (Å)")
plt.ylabel("Intensidad normalizada")

plt.title(f"Región {lambdai} - {lambdaf} Å Invierno")

plt.show()

In [ ]:
region_verano = verano[
    (verano["lambda"] > lambdai) &
    (verano["lambda"] < lambdaf)
]

region_invierno = invierno[
    (invierno["lambda"] > lambdai) &
    (invierno["lambda"] < lambdaf)
]

plt.figure(figsize=(10,4))

plt.plot(
    region_invierno["lambda"],
    region_invierno["intensity"],
    color="blue"
)

plt.plot(
    region_verano["lambda"],
    region_verano["intensity"],
    color="red"
)

plt.xlabel("Longitud de onda (Å)")
plt.ylabel("Intensidad normalizada")
plt.legend(["Invierno", "Verano"])

plt.title(f"Región {lambdai} - {lambdaf} Å Superpuestos")

plt.show()

In [ ]:
c = 299792.458

invierno_corregido = invierno.copy()
invierno_corregido["wavenumber"] = invierno["wavenumber"] - (invierno["wavenumber"]* (45 / c))
invierno_corregido['intensity'] = invierno['intensity']
invierno_corregido['lambda'] = 1e8 / invierno_corregido['wavenumber']

invierno_corregido = invierno_corregido.sort_values("lambda")
invierno_corregido = invierno_corregido.reset_index(drop=True)

region_verano = verano[
    (verano["lambda"] > lambdai) &
    (verano["lambda"] < lambdaf)
]

region_invierno_corregido = invierno_corregido[
    (invierno_corregido["lambda"] > lambdai) &
    (invierno_corregido["lambda"] < lambdaf)
]

plt.figure(figsize=(10,4))

plt.plot(
    region_invierno_corregido["lambda"],
    region_invierno_corregido["intensity"],
    color="grey"
)

plt.plot(
    region_verano["lambda"],
    region_verano["intensity"],
    color="black"
)

plt.xlabel("Longitud de onda (Å)")
plt.ylabel("Intensidad normalizada")
plt.legend(["Invierno corregido", "Verano"])
plt.title(f"Región {lambdai} - {lambdaf} Å Superpuestos y Corregidos")

plt.show()

Hay que corregir los espectros. Hinkle reporta un corrimiento relativo entre ambas fechas de $$45\frac{km}{s}$$

Entonces usamos para número de onda:

$$\
\Delta \sigma = \sigma \frac{45}{c}
\$$

In [ ]:
file = Nave.extractfile("./table2.gz")
with gzip.open(file,'rt') as f:

    lines = pd.read_fwf(
        f,
        colspecs=[
            (0,4),      # Num
            (5,15),     # Term1
            (15,16),    # -
            (16,26),    # Term2
            (27,28),    # J1
            (28,29),    # -
            (29,30),    # J2
            (30,36),    # I
            (37,47),    # Lvac
            (48,58),    # Lair
            (59,65),    # Lo-R
            (65,75),    # Sigma
            (76,77),    # q_Sigma
            (79,83),    # So-R
            (84,89),    # El
            (90,95),    # Eu
            (96,105),   # Elc
            (106,115),  # Euc
            (116,122),  # loggf
            (123,124),  # source
            (126,131)   # Blend
        ]
    )

lines.columns = [
"Num","Term1","dash1","Term2","J1","dash2","J2",
"I","Lvac","Lair","LoR","Sigma","qSigma","SoR",
"El","Eu","Elc","Euc","loggf","source","Blend"
]

 En el catálogo dice que las unidades de Lvac están en 0.1nm = 1Å, así que no hay que convertir nada. Usamos Lvacío porque con los datos de Hinkle se usó un interferómetro de transformada de Fourier.

In [ ]:
lines["lambda"] = lines["Lvac"]

fe_lines = lines[
    lines["qSigma"] == "A"
]

lambda_min = 10000
lambda_max = 10100

copy_fe_lines = fe_lines.copy()
copy_fe_lines = copy_fe_lines[
    (copy_fe_lines["lambda"] > lambda_min) &
    (copy_fe_lines["lambda"] < lambda_max)
]

espectro = verano.copy()
espectro = espectro[
    (espectro["lambda"] > lambda_min) &
    (espectro["lambda"] < lambda_max)
]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True, gridspec_kw={'height_ratios': [3, 1]})


ax1.plot(espectro["lambda"], espectro["intensity"], color="black")
ax1.set_ylabel("Flujo normalizado")
ax1.set_title(f"Espectro y líneas de Fe I ({lambda_min} - {lambda_max} Å)")


ax2.vlines(
    copy_fe_lines["lambda"],
    ymin=0,
    ymax=1,
    color="red"
)
ax2.set_ylabel("Líneas")
ax2.set_xlabel("Longitud de onda (Å)")
ax2.set_yticks([])


plt.tight_layout()
plt.subplots_adjust(hspace=0)
plt.show()

In [ ]:
c = 299792.458   # velocidad de la luz en km/s

# --- Parámetros de la detección (se pueden mover y volver a correr) ---
profundidad_minima = 0.10   # una línea menos profunda que esto la tomamos como ruido
tolerancia = 3              # km/s: qué tan lejos de la velocidad local aceptamos el mínimo
separacion_blend = 9        # km/s: si hay otra línea de Fe I más cerca que esto, es blend
ancho_ala = 25              # km/s: hasta dónde miramos a cada lado del mínimo
ruido = 0.015               # cuánto puede bajar el flujo por ruido sin que termine el ala

# --- Filtros de forma (para quedarnos solo con líneas limpias, no mezcladas) ---
tope_minimo = 0.94          # las dos alas deben subir al menos hasta aquí (si no, hay otra cosa)
ala_minima = 6              # km/s: cada ala debe medir al menos esto antes de cortarse
ancho_minimo = 5.0          # km/s: ancho a media altura (más angosta = ruido)
ancho_maximo = 11.5         # km/s: más ancha = varias líneas pegadas
cociente_alas_min = 0.6     # ancho izquierdo / ancho derecho (a media altura)
cociente_alas_max = 1.7     # fuera de este rango la línea es muy asimétrica = mezcla
curvatura_relativa_max = 0.2   # curvatura (en (km/s)^-2) / profundidad: más que esto es una punta rara
rugosidad_maxima = 0.5      # km/s: cuánto puede zigzaguear la bisectriz (hombros, micro estructura)
diferencia_mitades_max = 0.15   # al doblar el valle por el centro, las dos alas deben parecerse
meseta_maxima = 3.0         # km/s: largo máximo de un tramo plano en el ala (hombro = línea débil pegada)
meseta_margen = 0.03        # el tramo plano cuenta solo si está así de por debajo del continuo
meseta_pendiente = 0.12     # "plano" = pendiente menor que esta fracción de la pendiente máxima del ala

fraccion_bisectriz = 0.85   # la bisectriz sube hasta esta fracción de la profundidad
                            # (arriba el ala es casi plana y el ruido la desordena)

# --- Dónde buscamos líneas ---
lambda_busqueda_min = 9130
lambda_busqueda_max = 50000

# catalogo_A: solo calidad A (lo que pide el director)
# catalogo_todas: todas las calidades, solo lo usamos para saber si una línea tiene vecinas
catalogo_A = fe_lines[
    (fe_lines["Lvac"] > lambda_busqueda_min) &
    (fe_lines["Lvac"] < lambda_busqueda_max)
]
catalogo_todas = lines[
    (lines["Lvac"] > lambda_busqueda_min) &
    (lines["Lvac"] < lambda_busqueda_max)
]

print("Líneas de Fe I calidad A en el rango:", len(catalogo_A))
print("Líneas de Fe I de todas las calidades:", len(catalogo_todas))

In [ ]:
# Buscamos todos los mínimos del espectro y vemos a qué velocidad quedan
# respecto a las líneas del catálogo. El pico del histograma es el corrimiento típico.

epocas = {"Verano": verano, "Invierno": invierno}
colores = {"Verano": "red", "Invierno": "blue"}

plt.figure(figsize=(10, 4))

for nombre in epocas:
    espectro = epocas[nombre]
    lam = espectro["lambda"].values
    flujo = espectro["intensity"].fillna(1).values      # los huecos valen 1 (continuo)
    flujo_suave = savgol_filter(flujo, 7, 2)
    picos, _ = find_peaks(-flujo_suave, prominence=0.05)

    velocidades = []
    for l0 in catalogo_A["Lvac"]:
        v = c * (lam[picos] - l0) / l0
        cerca = v[np.abs(v + 10) < 80]
        velocidades.extend(cerca)

    plt.hist(velocidades, bins=np.arange(-90, 70, 1), alpha=0.5,
             color=colores[nombre], label=nombre)

plt.xlabel("Velocidad respecto al catálogo (km/s)")
plt.ylabel("Número de mínimos")
plt.title("¿Dónde caen los mínimos respecto a las líneas del catálogo?")
plt.legend()
plt.show()

# Mirando el histograma elegimos el centro de cada pico:
v_esperada_verano = 13
v_esperada_invierno = -29

In [ ]:
def buscar_ala(flujo, lam, i_min, sentido, ancho_max):
    # Camina desde el mínimo hacia afuera (sentido = -1 izquierda, +1 derecha)
    # mientras el flujo siga subiendo. Devuelve el índice donde termina el ala.
    i = i_min
    i_tope = i_min
    while True:
        j = i + sentido
        if j < 0 or j >= len(flujo):
            break
        if np.isnan(flujo[j]):
            return None                                  # hay un hueco: descartamos
        if abs(lam[j] - lam[i_min]) > ancho_max:
            break                                        # ya nos alejamos demasiado
        if flujo[j] < flujo[i_tope] - ruido:
            break                                        # el flujo empezó a bajar: otra línea
        if flujo[j] > flujo[i_tope]:
            i_tope = j
        i = j
    return i_tope


def cruces_en_nivel(lam, flujo, i_min, f1):
    # Para un nivel de flujo f1 busca, a cada lado del mínimo, dónde el espectro cruza f1.
    # Interpola linealmente entre el punto de arriba (N+) y el de abajo (N-).
    i = i_min
    while flujo[i] < f1:
        i = i - 1
    lambda_izq = lam[i] + (f1 - flujo[i]) * (lam[i + 1] - lam[i]) / (flujo[i + 1] - flujo[i])

    i = i_min
    while flujo[i] < f1:
        i = i + 1
    lambda_der = lam[i - 1] + (f1 - flujo[i - 1]) * (lam[i] - lam[i - 1]) / (flujo[i] - flujo[i - 1])

    return lambda_izq, lambda_der


def detectar_lineas(espectro, v_esperada, guias, profundidad_minima, tolerancia, filtrar_forma):
    lam = espectro["lambda"].values
    flujo = espectro["intensity"].values

    # Todos los mínimos del espectro (con un suavizado pequeño para no pescar ruido)
    flujo_lleno = np.where(np.isnan(flujo), 1.0, flujo)
    flujo_suave = savgol_filter(flujo_lleno, 7, 2)
    picos, _ = find_peaks(-flujo_suave, prominence=profundidad_minima / 2)

    lam_vecinas = np.sort(catalogo_todas["Lvac"].values)
    encontradas = []

    for _, linea in catalogo_A.iterrows():
        l0 = linea["Lvac"]

        # 1) Blend: ¿hay otra línea de Fe I demasiado cerca en el catálogo?
        otras = lam_vecinas[np.abs(lam_vecinas - l0) > 1e-6]
        distancia_v = np.abs(c * (otras - l0) / l0)
        if distancia_v.min() < separacion_blend:
            continue

        # 2) Velocidad donde esperamos el mínimo
        v_centro = v_esperada
        if guias is not None:
            # mediana de las 7 líneas guía más cercanas en longitud de onda
            orden = np.argsort(np.abs(guias["Lvac"].values - l0))[:7]
            v_centro = np.median(guias["v"].values[orden])

        # 3) Mínimo del espectro más cercano a lo esperado
        v_picos = c * (lam[picos] - l0) / l0
        cerca = np.where(np.abs(v_picos - v_centro) < tolerancia)[0]
        if len(cerca) == 0:
            continue
        mejor = cerca[np.argmin(np.abs(v_picos[cerca] - v_centro))]
        i_min = picos[mejor]
        if np.isnan(flujo[i_min]):
            continue

        # 4) Alas a cada lado (sin huecos)
        ancho = ancho_ala / c * lam[i_min]
        i_izq = buscar_ala(flujo_suave, lam, i_min, -1, ancho)
        i_der = buscar_ala(flujo_suave, lam, i_min, +1, ancho)
        if i_izq is None or i_der is None:
            continue
        if np.isnan(flujo[i_izq:i_der + 1]).any():
            continue

        # 5) Profundidad medida desde el tope más bajo de las dos alas
        tope = min(flujo[i_izq], flujo[i_der])
        profundidad = tope - flujo[i_min]
        if profundidad < profundidad_minima:
            continue

        # 6) Forma de la línea: solo en el paso final (en las guías no hace falta)
        lambda_izq, lambda_der = cruces_en_nivel(lam, flujo, i_min, flujo[i_min] + 0.5 * profundidad)
        ancho_izq = c * (lam[i_min] - lambda_izq) / lam[i_min]
        ancho_der = c * (lambda_der - lam[i_min]) / lam[i_min]
        ancho = ancho_izq + ancho_der
        cociente_alas = ancho_izq / ancho_der
        ala_izq_km = c * (lam[i_min] - lam[i_izq]) / lam[i_min]
        ala_der_km = c * (lam[i_der] - lam[i_min]) / lam[i_min]

        if filtrar_forma:
            if tope < tope_minimo:
                continue
            if ala_izq_km < ala_minima or ala_der_km < ala_minima:
                continue
            if ancho < ancho_minimo or ancho > ancho_maximo:
                continue
            if cociente_alas < cociente_alas_min or cociente_alas > cociente_alas_max:
                continue

        encontradas.append({
            "Lvac": l0,
            "v": c * (lam[i_min] - l0) / l0,
            "i_min": i_min,
            "i_izq": i_izq,
            "i_der": i_der,
            "profundidad": profundidad,
            "tope": tope,
            "ancho": ancho,
            "cociente_alas": cociente_alas,
            "El": linea["El"],
        })

    return pd.DataFrame(encontradas)


def detectar(espectro, v_esperada):
    # Paso 1: solo líneas fuertes y ventana ancha. Son las "guías".
    guias = detectar_lineas(espectro, v_esperada, None, 0.30, 6, False)
    # Paso 2: todas las líneas, con ventana angosta alrededor de la velocidad
    # que tienen las guías cercanas (el cero del atlas cambia de un tramo a otro).
    return detectar_lineas(espectro, v_esperada, guias, profundidad_minima, tolerancia, True)


detectadas_verano = detectar(verano, v_esperada_verano)
detectadas_invierno = detectar(invierno, v_esperada_invierno)

print("Líneas detectadas en verano:", len(detectadas_verano))
print("Líneas detectadas en invierno:", len(detectadas_invierno))

In [ ]:
def elegir_numero_de_puntos(profundidad):
    # Entre 5 y 8 puntos: las líneas débiles tienen más ruido y piden más puntos
    if profundidad > 0.40:
        return 7
    if profundidad > 0.25:
        return 7
    if profundidad > 0.15:
        return 7
    return 7


def ajustar_cabecita(lam, flujo, i_min, profundidad):
    # Polinomio de grado 4 con los n puntos más cercanos a la punta del valle.
    # Primero calculamos dónde está la punta, y luego repetimos el ajuste eligiendo los puntos
    # más cercanos a esa punta, así quedan repartidos parejo a cada lado.
    n = elegir_numero_de_puntos(profundidad)
    if i_min - 8 < 0 or i_min + 9 > len(lam):
        return None
    candidatos = np.arange(i_min - 8, i_min + 9)
    x0 = lam[i_min]                              # restamos x0 para que los números sean chicos
    punta = lam[i_min]                           # primera idea de dónde está la punta

    for repeticion in range(3):
        distancia = np.abs(lam[candidatos] - punta)
        elegidos = np.sort(candidatos[np.argsort(distancia)[:n]])
        x = lam[elegidos]
        y = flujo[elegidos]
        if np.isnan(y).any():
            return None

        poli = Polynomial.fit(x - x0, y, 4)

        # Mínimo del polinomio: lo buscamos evaluándolo en muchos puntos
        xs = np.linspace(x[0] - x0, x[-1] - x0, 2001)
        k = np.argmin(poli(xs))
        punta = x0 + xs[k]

    return {
        "n": n,
        "x": x,
        "y": y,
        "poli": poli,
        "x0": x0,
        "lambda_min": punta,
        "flujo_centro": poli(xs[k]),
        "f2": poli.deriv(2)(xs[k]),              # segunda derivada respecto a lambda
        "f3": poli.deriv(3)(xs[k]),              # tercera derivada respecto a lambda
    }


def calcular_bisectriz(lam, flujo, i_izq, i_min, i_der):
    # Para cada nivel de flujo f1 buscamos dónde cruza cada ala y promediamos los dos puntos.
    f_min = flujo[i_min]
    tope_alas = min(flujo[i_izq], flujo[i_der])
    f_tope = f_min + fraccion_bisectriz * (tope_alas - f_min)
    niveles = np.linspace(f_min, f_tope, 102)[1:-1]      # 100 niveles entre punta y tope

    bis_flujo = []
    bis_lambda = []
    for f1 in niveles:
        lambda_izq, lambda_der = cruces_en_nivel(lam, flujo, i_min, f1)
        bis_flujo.append(f1)
        bis_lambda.append(0.5 * (lambda_izq + lambda_der))

    return np.array(bis_flujo), np.array(bis_lambda)


def comparar_mitades(lam, flujo, i_izq, i_min, i_der, lambda_min, profundidad):
    # Doblamos el valle por el centro y comparamos el ala izquierda con la derecha
    # a las mismas distancias del centro (en km/s). Un valle limpio se parece a sí mismo;
    # uno deformado (hombros, mezclas) no. Devuelve la diferencia típica en fracción de la profundidad.
    distancia_izq = (c * (lambda_min - lam[i_izq:i_min + 1]) / lambda_min)[::-1]
    flujo_izq = flujo[i_izq:i_min + 1][::-1]
    distancia_der = c * (lam[i_min:i_der + 1] - lambda_min) / lambda_min
    flujo_der = flujo[i_min:i_der + 1]

    hasta = min(distancia_izq[-1], distancia_der[-1])
    distancias = np.linspace(0.5, hasta, 40)
    diferencia = np.interp(distancias, distancia_izq, flujo_izq) - np.interp(distancias, distancia_der, flujo_der)
    return np.sqrt(np.mean(diferencia ** 2)) / profundidad


def largo_de_meseta(pendiente, flujo, distancia, tope):
    # Recorre un ala hacia afuera desde donde sube más rápido y busca tramos casi planos
    # que están claramente por debajo del continuo. Devuelve el largo del más largo (km/s).
    k = np.argmax(pendiente)
    limite = meseta_pendiente * pendiente.max()
    largo = 0
    inicio = None
    for i in range(k, len(pendiente)):
        es_plano = abs(pendiente[i]) < limite and flujo[i] < tope - meseta_margen
        if es_plano:
            if inicio is None:
                inicio = i
            largo = max(largo, distancia[i] - distancia[inicio])
        else:
            inicio = None
    return largo


def buscar_mesetas(lam, flujo, i_izq, i_min, i_der, lambda_min):
    # Un hombro en el ala se ve como una meseta: el flujo deja de subir un rato
    # y después sigue subiendo. Es la firma de una línea débil pegada a la principal.
    if i_izq < 3 or i_der + 4 > len(flujo):
        return 0
    trozo = np.where(np.isnan(flujo[i_izq - 3:i_der + 4]), 1.0, flujo[i_izq - 3:i_der + 4])
    paso = np.median(np.diff(lam[i_izq:i_der + 1]))
    pendiente = savgol_filter(trozo, 7, 2, deriv=1, delta=paso)[3:-3]     # derivada respecto a lambda
    suave = savgol_filter(trozo, 5, 2)[3:-3]
    tope = min(flujo[i_izq], flujo[i_der])
    centro = i_min - i_izq

    # ala izquierda (la recorremos hacia afuera, por eso la damos vuelta)
    pendiente_izq = -pendiente[:centro + 1][::-1]
    flujo_izq = suave[:centro + 1][::-1]
    distancia_izq = c * (lambda_min - lam[i_izq:i_min + 1][::-1]) / lambda_min
    # ala derecha
    pendiente_der = pendiente[centro:]
    flujo_der = suave[centro:]
    distancia_der = c * (lam[i_min:i_der + 1] - lambda_min) / lambda_min

    largo_izq = largo_de_meseta(pendiente_izq, flujo_izq, distancia_izq, tope)
    largo_der = largo_de_meseta(pendiente_der, flujo_der, distancia_der, tope)
    return max(largo_izq, largo_der)


def construir_tabla(espectro, detectadas):
    lam = espectro["lambda"].values
    flujo = espectro["intensity"].values
    filas = []

    for _, d in detectadas.iterrows():
        ajuste = ajustar_cabecita(lam, flujo, int(d["i_min"]), d["profundidad"])
        if ajuste is None:
            continue

        l0 = d["Lvac"]
        lambda_min = ajuste["lambda_min"]
        curvatura = ajuste["f2"] * lambda_min ** 2         # segunda derivada por lambda al cuadrado
        asimetria = ajuste["f3"]                           # tercera derivada

        # Solo para el filtro: la curvatura en (km/s)^-2 dividida por la profundidad
        curvatura_relativa = ajuste["f2"] * (lambda_min / c) ** 2 / d["profundidad"]
        if curvatura_relativa > curvatura_relativa_max:
            continue                                   # punta rara: la descartamos

        # Rugosidad de la bisectriz: cuánto se aleja de una curva suave.
        # Un hombro o una mezcla en las alas la hace zigzaguear.
        bis_flujo, bis_lambda = calcular_bisectriz(lam, flujo, int(d["i_izq"]), int(d["i_min"]), int(d["i_der"]))
        bis_velocidad = c * (bis_lambda - lambda_min) / lambda_min
        curva_suave = np.polyfit(bis_flujo, bis_velocidad, 2)
        rugosidad = np.abs(bis_velocidad - np.polyval(curva_suave, bis_flujo)).max()
        if rugosidad > rugosidad_maxima:
            continue

        # Forma del valle: ¿las dos alas se parecen?
        diferencia_mitades = comparar_mitades(lam, flujo, int(d["i_izq"]), int(d["i_min"]),
                                              int(d["i_der"]), lambda_min, d["profundidad"])
        if diferencia_mitades > diferencia_mitades_max:
            continue

        # Hombros: tramos planos en las alas
        meseta = buscar_mesetas(lam, flujo, int(d["i_izq"]), int(d["i_min"]), int(d["i_der"]), lambda_min)
        if meseta > meseta_maxima:
            continue
        filas.append({
            "lambda_catalogo": l0,
            "lambda_min": lambda_min,
            "velocidad": c * (lambda_min - l0) / l0,                 # Doppler, km/s
            "flujo_centro": ajuste["flujo_centro"],
            "curvatura": curvatura,
            "asimetria": asimetria,
            "potencial_excitacion": d["El"],
            "numero_puntos": ajuste["n"],
            "profundidad": d["profundidad"],
            "ancho_km_s": d["ancho"],
            "cociente_alas": d["cociente_alas"],
            "rugosidad_bisectriz": rugosidad,
            "diferencia_mitades": diferencia_mitades,
            "meseta_km_s": meseta,
            "i_min": int(d["i_min"]),
            "i_izq": int(d["i_izq"]),
            "i_der": int(d["i_der"]),
        })

    return pd.DataFrame(filas)


resultados_verano = construir_tabla(verano, detectadas_verano)
resultados_invierno = construir_tabla(invierno, detectadas_invierno)

display(resultados_verano.head())
print("Verano:", len(resultados_verano), "líneas  |  Invierno:", len(resultados_invierno), "líneas")

## Comparación de atípicas (alternativa a los cortes duros de forma)

Idea: en vez de decidir con umbrales fijos (`ancho_maximo`, `rugosidad_maxima`, ...), hacemos así:

1. **Primera detección** con los umbrales de siempre (`detectadas_*`). Esas líneas son el *núcleo*: nos dicen cómo se ve una línea típica.
2. **Medimos** la forma de *todas* las candidatas (también las que los cortes de forma habrían tirado): curvatura, asimetría, ancho, cociente de alas, rugosidad, simetría de mitades y mesetas.
3. **Comparamos métrica por métrica** con lo típico del núcleo (mediana y MAD, recortando iterativamente las rarezas). Para las métricas que cambian con la profundidad (ancho, curvatura...) lo típico es una *curva según la profundidad*, porque las líneas profundas son naturalmente más anchas. Si una línea se aleja más de `k_atipico` desviaciones robustas en alguna, es **atípica**.
4. **Comparamos por combinación** (distancia de Mahalanobis robusta): atrapa líneas que no se pasan en ninguna métrica por separado pero son raras en conjunto.
5. **Mostramos** las atípicas y **por qué** lo son.
6. Las que no son atípicas son las **líneas limpias**.

**Ojo con el sesgo:** "lo típico" lo define el núcleo, que sale de tus umbrales. Este método detecta lo que *se aleja de lo que tus umbrales aceptaron*, no lo que es "bueno" en un sentido absoluto. Conviene revisar a ojo la galería de atípicas y la de limpias.

Asimetría positiva = ala azul más extendida; negativa = ala roja más extendida.

### 1. Detección permisiva y medición de todas las candidatas

In [ ]:
# ============================================================
# COMPARACIÓN DE ATÍPICAS (reemplaza los cortes duros de forma)
# ============================================================
import textwrap

# --- Parámetros (se pueden mover y volver a correr) ---
k_atipico = 3.27           # cuántas "desviaciones robustas" puede alejarse una línea de lo típico
grado_tendencia = 2        # grado de la curva "lo típico según la profundidad" (1 = recta, 2 = parábola)
nivel_combinacion = 0.999  # percentil chi-cuadrado para la distancia robusta (más alto = más permisivo)
usar_comparacion = True    # True: al final, resultados_* pasan a ser las líneas limpias de este método

# Qué comparamos: columna -> (transformación, qué lados vigilamos, escala mínima)
# La escala mínima evita que una métrica casi constante (la meseta casi siempre vale 0)
# marque como rara cualquier diferencia diminuta.
vigiladas = {
    "curv_adim":           ("lineal", "ambos", 0.30),
    "asim_adim":           ("lineal", "ambos", 0.50),
    "ancho_km_s":          ("lineal", "ambos", 0.30),
    "cociente_alas":       ("log",    "ambos", 0.05),
    "rugosidad_bisectriz": ("log",    "alto",  0.10),
    "diferencia_mitades":  ("log",    "alto",  0.10),
    "meseta_km_s":         ("lineal", "alto",  0.50),
    # Curvatura y asimetría "crudas" (como las ves en tus gráficas), divididas solo por la profundidad.
    # Las adimensionales de arriba quitan también el ancho, y por eso una línea profunda Y angosta
    # podía pasar de largo: los dos efectos se cancelaban.
    "curv_prof":           ("lineal", "ambos", 1.00),
    "asim_prof":           ("lineal", "ambos", 10.0),
}

# Métricas cuyo valor típico CAMBIA con la profundidad (las líneas profundas son naturalmente más anchas y
# de punta menos aguda). Para estas, "lo típico" es una curva según la profundidad y no un solo número;
# si no, castigaríamos a las líneas profundas solo por serlo.
con_tendencia = ["curv_adim", "ancho_km_s", "curv_prof", "diferencia_mitades"]

# Cómo se lee cada bandera (alto = por encima de lo típico, bajo = por debajo)
explicacion = {
    ("curv_adim", "alto"):           "punta muy aguda",
    ("curv_adim", "bajo"):           "punta muy achatada",
    ("asim_adim", "alto"):           "ala AZUL mucho más extendida",
    ("asim_adim", "bajo"):           "ala ROJA mucho más extendida",
    ("ancho_km_s", "alto"):          "línea muy ancha (¿varias pegadas?)",
    ("ancho_km_s", "bajo"):          "línea muy angosta (¿ruido?)",
    ("cociente_alas", "alto"):       "ala izquierda (azul) mucho más ancha que la derecha",
    ("cociente_alas", "bajo"):       "ala derecha (roja) mucho más ancha que la izquierda",
    ("rugosidad_bisectriz", "alto"): "bisectriz zigzaguea (hombro o micro-estructura)",
    ("diferencia_mitades", "alto"):  "las dos alas no se parecen al doblar el valle",
    ("meseta_km_s", "alto"):         "tramo plano en el ala (hombro: línea débil pegada)",
    ("curv_prof", "alto"):           "curvatura cruda muy alta para su profundidad",
    ("curv_prof", "bajo"):           "curvatura cruda muy baja para su profundidad",
    ("asim_prof", "alto"):           "asimetría cruda muy positiva para su profundidad (ala azul)",
    ("asim_prof", "bajo"):           "asimetría cruda muy negativa para su profundidad (ala roja)",
}

# Ingredientes de la comparación por combinación: las z de estas métricas (ya corregidas por profundidad,
# así que una línea profunda no sale rara solo por serlo) y cómo se llaman en el motivo.
ingredientes = [
    ("curv_adim",           "curvatura de la punta"),
    ("asim_adim",           "asimetría de la punta"),
    ("ancho_km_s",          "ancho"),
    ("cociente_alas",       "cociente de alas"),
    ("rugosidad_bisectriz", "rugosidad de la bisectriz"),
    ("diferencia_mitades",  "diferencia entre mitades"),
]


def detectar_permisivo(espectro, v_esperada):
    # Igual que detectar(), pero sin los filtros de forma (ancho, cociente de alas, tope, alas mínimas).
    # Sigue pidiendo: no blend, profundidad mínima, mínimo cerca de la velocidad esperada, sin huecos.
    guias = detectar_lineas(espectro, v_esperada, None, 0.30, 6, False)
    return detectar_lineas(espectro, v_esperada, guias, profundidad_minima, tolerancia, False)


def calcular_candidatas(espectro, permisivas, estrictas):
    # Igual que construir_tabla(), pero NO descarta nada: mide todo y deja la decisión para después.
    # "en_nucleo" marca las que ya habían pasado la primera detección con umbrales.
    lam = espectro["lambda"].values
    flujo = espectro["intensity"].values
    nucleo = set(np.round(estrictas["Lvac"].values, 4))
    filas = []

    for _, d in permisivas.iterrows():
        ajuste = ajustar_cabecita(lam, flujo, int(d["i_min"]), d["profundidad"])
        if ajuste is None:
            continue
        l0 = d["Lvac"]
        lambda_min = ajuste["lambda_min"]
        i_min, i_izq, i_der = int(d["i_min"]), int(d["i_izq"]), int(d["i_der"])

        bis_flujo, bis_lambda = calcular_bisectriz(lam, flujo, i_izq, i_min, i_der)
        bis_velocidad = c * (bis_lambda - lambda_min) / lambda_min
        curva_suave = np.polyfit(bis_flujo, bis_velocidad, 2)
        rugosidad = np.abs(bis_velocidad - np.polyval(curva_suave, bis_flujo)).max()

        # Curvatura y asimetría "adimensionales": dividimos por la profundidad y por el ancho
        # de la línea (km/s). Así miden solo la FORMA de la punta, no qué tan angosta o profunda es
        # (el ancho y la profundidad ya se comparan por su cuenta).
        # Referencia: una gaussiana perfecta da curv_adim = 8 ln2 = 5.55 y asim_adim = 0.
        ancho = d["ancho"]
        curv_adim = ajuste["f2"] * (lambda_min / c) ** 2 / d["profundidad"] * ancho ** 2
        asim_adim = ajuste["f3"] * (lambda_min / c) ** 3 / d["profundidad"] * ancho ** 3

        filas.append({
            "lambda_catalogo": l0,
            "lambda_min": lambda_min,
            "velocidad": c * (lambda_min - l0) / l0,
            "flujo_centro": ajuste["flujo_centro"],
            "profundidad": d["profundidad"],
            "tope": d["tope"],
            "ancho_km_s": ancho,
            "cociente_alas": d["cociente_alas"],
            "curvatura": ajuste["f2"] * lambda_min ** 2,       # igual que en tu tabla original
            "asimetria": ajuste["f3"],                          # igual que en tu tabla original
            "curv_prof": ajuste["f2"] * lambda_min ** 2 / d["profundidad"] / 1e9,   # curvatura cruda / profundidad
            "asim_prof": ajuste["f3"] / d["profundidad"],                            # asimetría cruda / profundidad
            "curv_adim": curv_adim,
            "asim_adim": asim_adim,
            "rugosidad_bisectriz": rugosidad,
            "diferencia_mitades": comparar_mitades(lam, flujo, i_izq, i_min, i_der, lambda_min, d["profundidad"]),
            "meseta_km_s": buscar_mesetas(lam, flujo, i_izq, i_min, i_der, lambda_min),
            "potencial_excitacion": d["El"],
            "numero_puntos": ajuste["n"],
            "i_min": i_min, "i_izq": i_izq, "i_der": i_der,
            "en_nucleo": round(l0, 4) in nucleo,
        })
    return pd.DataFrame(filas)


def medir_z(tabla, referencia_mascara):
    # Para cada métrica: centro = lo típico (mediana de la referencia, o su curva según la profundidad),
    # escala = MAD de lo que se aleja la referencia (no se deja arrastrar por extremos).
    # z = cuántas desviaciones robustas se aleja cada línea de su centro.
    profundidad = tabla["profundidad"].values
    z, referencia, centros = {}, {}, {}
    for col, (transformacion, lados, escala_minima) in vigiladas.items():
        x = tabla[col].values.astype(float)
        if transformacion == "log":
            x = np.log(x)

        if col in con_tendencia:
            usar = referencia_mascara.copy()
            for _ in range(4):                       # ajuste robusto: ajusta, quita lo que se aleja, repite
                coef = np.polyfit(profundidad[usar], x[usar], grado_tendencia)
                residuo = x - np.polyval(coef, profundidad)
                esc = 1.4826 * np.median(np.abs(residuo[referencia_mascara]))
                usar = referencia_mascara & (np.abs(residuo) < 3 * max(esc, 1e-9))
            # No extrapolamos: fuera del rango de profundidades que cubre la referencia, usamos el borde
            # (una parábola se dispara donde casi no hay líneas con las que aprender).
            prof_evaluada = np.clip(profundidad, profundidad[usar].min(), profundidad[usar].max())
            centro = np.polyval(coef, prof_evaluada)
            residuo = x - centro
            escala = max(1.4826 * np.median(np.abs(residuo[usar])), escala_minima)
        else:
            centro = np.full(len(x), np.median(x[referencia_mascara]))
            escala = max(1.4826 * np.median(np.abs(x[referencia_mascara] - centro[0])), escala_minima)

        z[col] = (x - centro) / escala
        centros[col] = centro
        referencia[col] = (np.median(centro), escala, transformacion)
    return z, referencia, centros


def es_atipica(z):
    # True si alguna métrica se pasa de k_atipico (hacia arriba, o hacia ambos lados si así se vigila)
    marca = np.zeros(len(next(iter(z.values()))), dtype=bool)
    for col, (_, lados, _) in vigiladas.items():
        marca |= (z[col] > k_atipico) if lados == "alto" else (np.abs(z[col]) > k_atipico)
    return marca


def comparar_con_nucleo(tabla, iteraciones=5):
    # "Lo típico" se define con el núcleo (lo que pasó la primera detección). Como el núcleo todavía
    # puede traer mezclas escondidas, repetimos: marcamos atípicas, recalculamos lo típico SIN ellas
    # y volvemos a marcar, hasta que ya no cambie (recorte iterativo).
    nucleo = tabla["en_nucleo"].values
    referencia_mascara = nucleo.copy()
    for _ in range(iteraciones):
        z, referencia, centros = medir_z(tabla, referencia_mascara)
        atipica = es_atipica(z)
        nueva = nucleo & ~atipica
        if (nueva == referencia_mascara).all():
            break
        referencia_mascara = nueva

    tabla = tabla.copy()
    motivos = [[] for _ in range(len(tabla))]
    puntaje = np.zeros(len(tabla))
    for col, (transformacion, lados, _) in vigiladas.items():
        tipico = np.exp(centros[col]) if transformacion == "log" else centros[col]    # típico A ESA profundidad
        x = tabla[col].values.astype(float)
        tabla["z_" + col] = z[col]
        for i in range(len(tabla)):
            if z[col][i] > k_atipico:
                lado = "alto"
            elif z[col][i] < -k_atipico and lados == "ambos":
                lado = "bajo"
            else:
                continue
            motivos[i].append(f"{explicacion[(col, lado)]} ({x[i]:.2f} vs típico {tipico[i]:.2f})")
            puntaje[i] = max(puntaje[i], abs(z[col][i]))

    atipica_z = np.array([len(m) > 0 for m in motivos])

    # Comparación por COMBINACIÓN: una línea puede estar a ~2 desviaciones en cada métrica (nada grave
    # por separado) y aun así ser rarísima en conjunto, por ejemplo muy profunda Y muy angosta a la vez.
    # La distancia de Mahalanobis robusta mide eso, teniendo en cuenta cómo se mueven juntas las métricas.
    # (Trabaja con las z, que ya descuentan la profundidad.)
    tabla["d2_robusta"] = np.nan
    tabla["atipica_combinacion"] = False
    try:
        from sklearn.covariance import MinCovDet
        from scipy.stats import chi2
        X = np.column_stack([z[col] for col, _ in ingredientes])
        sana = nucleo & ~atipica_z                        # lo típico se aprende solo de líneas sanas
        modelo = MinCovDet(random_state=0).fit(X[sana])
        d2 = modelo.mahalanobis(X)
        corte = chi2.ppf(nivel_combinacion, X.shape[1])
        desv = np.sqrt(np.diag(modelo.covariance_))
        for i in np.where((d2 > corte) & ~atipica_z)[0]:
            partes = []
            for j, (_, nombre) in enumerate(ingredientes):
                zj = (X[i, j] - modelo.location_[j]) / desv[j]
                if abs(zj) > 1.5:
                    partes.append(f"{nombre} {'alta' if zj > 0 else 'baja'} ({zj:+.1f}σ)")
            motivos[i].append("combinación poco común: " + ", ".join(partes) +
                              f" [distancia robusta {d2[i]:.0f} vs corte {corte:.0f}]")
            puntaje[i] = max(puntaje[i], np.sqrt(d2[i] / corte) * k_atipico)
            tabla.loc[tabla.index[i], "atipica_combinacion"] = True
        tabla["d2_robusta"] = d2
    except ImportError:
        print("Aviso: no está scikit-learn, me salto la comparación por combinación (pip install scikit-learn).")

    tabla["motivos"] = ["; ".join(m) for m in motivos]
    tabla["atipica"] = [len(m) > 0 for m in motivos]
    tabla["puntaje"] = puntaje
    return tabla, pd.DataFrame(referencia, index=["centro", "escala", "transformacion"]).T


In [ ]:
permisivas_verano = detectar_permisivo(verano, v_esperada_verano)
permisivas_invierno = detectar_permisivo(invierno, v_esperada_invierno)

candidatas_verano = calcular_candidatas(verano, permisivas_verano, detectadas_verano)
candidatas_invierno = calcular_candidatas(invierno, permisivas_invierno, detectadas_invierno)

comparadas_verano, referencia_verano = comparar_con_nucleo(candidatas_verano)
comparadas_invierno, referencia_invierno = comparar_con_nucleo(candidatas_invierno)

for nombre, t in [("Verano", comparadas_verano), ("Invierno", comparadas_invierno)]:
    print(f"{nombre}: {len(t)} candidatas  |  núcleo (primera detección): {t['en_nucleo'].sum()}  |  "
          f"atípicas: {t['atipica'].sum()} (de ellas {t['atipica_combinacion'].sum()} solo por combinación)  |  "
          f"limpias: {(~t['atipica']).sum()}")

print("\nLo típico (calculado con el núcleo de verano):")
display(referencia_verano.assign(centro=lambda d: d["centro"].astype(float).round(3),
                                 escala=lambda d: d["escala"].astype(float).round(3)))


### 2. Mapa de forma: ¿dónde están las atípicas?

In [ ]:
# Mapa de forma: cada punto es una línea; los ejes son "cuántas desviaciones robustas se aleja de lo típico
# A SU PROFUNDIDAD". Dentro del recuadro gris = típica en curvatura y asimetría. Rojo = atípica (por esta u otra métrica).
fig, ejes = plt.subplots(1, 2, figsize=(13, 5))

for eje, (nombre, t) in zip(ejes, [("Verano", comparadas_verano), ("Invierno", comparadas_invierno)]):
    eje.axvspan(-k_atipico, k_atipico, color="grey", alpha=0.12)
    eje.axhspan(-k_atipico, k_atipico, color="grey", alpha=0.12)

    normales = t[~t["atipica"]]
    raras = t[t["atipica"]]
    eje.scatter(normales["z_curv_adim"], normales["z_asim_adim"], s=14, color="black", label=f"limpias ({len(normales)})")
    eje.scatter(raras["z_curv_adim"], raras["z_asim_adim"], s=22, color="red", label=f"atípicas ({len(raras)}), por esta u otra métrica")
    eje.axhline(0, color="grey", lw=0.8)
    eje.axvline(0, color="grey", lw=0.8)
    eje.set_xscale("symlog", linthresh=5)          # escala simlog: un punto extremo no aplasta al resto
    eje.set_yscale("symlog", linthresh=5)
    eje.set_xlabel("Curvatura de la punta (desviaciones robustas)\n−: punta achatada    +: punta aguda")
    eje.set_ylabel("Asimetría de la punta (desviaciones robustas)\n+: ala azul extendida   −: ala roja extendida")
    eje.set_title(nombre)
    eje.legend(fontsize=8)

plt.tight_layout()
plt.show()


### 3. Las atípicas y por qué lo son

In [ ]:
def mostrar_atipicas(espectro, tabla, nombre, max_lineas=24):
    # Una ventana por línea rara, ordenadas de la más rara a la menos rara.
    # El título dice POR QUÉ es rara.
    lam = espectro["lambda"].values
    flujo = espectro["intensity"].values
    raras = tabla[tabla["atipica"]].sort_values("puntaje", ascending=False).head(max_lineas)
    if len(raras) == 0:
        print(nombre, ": no hay atípicas")
        return

    columnas = 4
    filas = int(np.ceil(len(raras) / columnas))
    fig, ejes = plt.subplots(filas, columnas, figsize=(16, 4.2 * filas), squeeze=False)
    ejes = ejes.flatten()

    for eje, (_, linea) in zip(ejes, raras.iterrows()):
        i_min, i_izq, i_der = int(linea["i_min"]), int(linea["i_izq"]), int(linea["i_der"])
        eje.plot(lam[i_izq:i_der + 1], flujo[i_izq:i_der + 1], ".", color="grey", markersize=4)
        ajuste = ajustar_cabecita(lam, flujo, i_min, linea["profundidad"])
        eje.plot(ajuste["x"], ajuste["y"], "o", color="black", markersize=5)
        x_curva = np.linspace(ajuste["x"][0], ajuste["x"][-1], 100)
        eje.plot(x_curva, ajuste["poli"](x_curva - ajuste["x0"]), color="blue")
        eje.plot(linea["lambda_min"], linea["flujo_centro"], "o", color="green", markersize=7)
        eje.axvline(linea["lambda_catalogo"], color="red", alpha=0.6)

        motivo = "\n".join(textwrap.fill(m, 48, initial_indent="• ", subsequent_indent="   ")
                           for m in linea["motivos"].split("; "))
        eje.set_title(f"{linea['lambda_catalogo']:.2f} Å\n{motivo}", fontsize=7, loc="left", color="darkred")
        eje.ticklabel_format(useOffset=False)
        eje.xaxis.set_major_locator(plt.MaxNLocator(4))

    for eje in ejes[len(raras):]:
        eje.axis("off")

    fig.suptitle(f"{nombre}: las {len(raras)} atípicas más raras (de {tabla['atipica'].sum()}). "
                 f"Negro = puntos del ajuste, azul = polinomio, verde = mínimo, rojo = catálogo", y=1.0)
    plt.tight_layout()
    plt.show()


mostrar_atipicas(verano, comparadas_verano, "Verano")
mostrar_atipicas(invierno, comparadas_invierno, "Invierno")

# Tabla de atípicas con su motivo
with pd.option_context("display.max_colwidth", 140, "display.width", 250):
    for nombre, t in [("Verano", comparadas_verano), ("Invierno", comparadas_invierno)]:
        print(f"\n{nombre}: atípicas y por qué")
        display(t[t["atipica"]].sort_values("puntaje", ascending=False)[
            ["lambda_catalogo", "profundidad", "curv_adim", "asim_adim", "puntaje", "motivos"]].round(3).head(30))


### 4. Líneas limpias (lo que sigue usa estas)

In [ ]:
# Líneas limpias = candidatas que NO resultaron atípicas en ninguna comparación
usar_mahalanobis = False   # False: solo MAD (versión principal). True: MAD + Mahalanobis (versión anterior)
if usar_mahalanobis:
    limpias_verano = comparadas_verano[~comparadas_verano["atipica"]].reset_index(drop=True)
    limpias_invierno = comparadas_invierno[~comparadas_invierno["atipica"]].reset_index(drop=True)
else:
    # se deja pasar lo que SOLO había rechazado la comparación por combinación (Mahalanobis)
    limpias_verano = comparadas_verano[~comparadas_verano["atipica"] | comparadas_verano["atipica_combinacion"]].reset_index(drop=True)
    limpias_invierno = comparadas_invierno[~comparadas_invierno["atipica"] | comparadas_invierno["atipica_combinacion"]].reset_index(drop=True)
print("Líneas limpias  |  Verano:", len(limpias_verano), " Invierno:", len(limpias_invierno))

if usar_comparacion:
    # Las celdas de abajo (gráficas y Excel) usan resultados_*: les damos las limpias de este método
    resultados_verano_umbrales = resultados_verano
    resultados_invierno_umbrales = resultados_invierno
    resultados_verano = limpias_verano
    resultados_invierno = limpias_invierno
    print("resultados_verano y resultados_invierno ahora son las líneas limpias por comparación "
          "(las anteriores quedaron en resultados_*_umbrales).")


### 5. Verificación sobre las gráficas crudas

In [ ]:
# Verificación: tus mismas gráficas (curvatura cruda vs flujo), con las atípicas en rojo.
# Si el método funciona, lo que "vuela" debería ser rojo y lo limpio debería formar una nube compacta.
fig, ejes = plt.subplots(2, 2, figsize=(13, 9))

for fila, (nombre, t) in enumerate([("Verano", comparadas_verano), ("Invierno", comparadas_invierno)]):
    limpias_t = t[~t["atipica"]]
    raras_t = t[t["atipica"]]

    eje = ejes[fila, 0]
    eje.scatter(raras_t["flujo_centro"], raras_t["curvatura"], s=14, color="red", alpha=0.5, label=f"atípicas ({len(raras_t)})")
    eje.scatter(limpias_t["flujo_centro"], limpias_t["curvatura"], s=14, color="black", label=f"limpias ({len(limpias_t)})")
    eje.set_xlabel("Flujo en el centro")
    eje.set_ylabel("Curvatura (f'' · λ²)")
    eje.set_title(f"{nombre}: curvatura cruda")
    eje.legend(fontsize=8)

    eje = ejes[fila, 1]
    eje.scatter(raras_t["flujo_centro"], raras_t["asimetria"], s=14, color="red", alpha=0.5, label="atípicas")
    eje.scatter(limpias_t["flujo_centro"], limpias_t["asimetria"], s=14, color="black", label="limpias")
    eje.axhline(0, color="grey", lw=0.8)
    eje.set_yscale("symlog", linthresh=10)
    eje.set_xlabel("Flujo en el centro")
    eje.set_ylabel("Asimetría (f''')")
    eje.set_title(f"{nombre}: asimetría cruda")
    eje.legend(fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
lambda_min = 10420
lambda_max = 10520

# False: las líneas rojas están donde dice el catálogo (en reposo).
# True: las líneas rojas se corren con la velocidad de cada época, para verlas sobre los mínimos.
desplazar_catalogo = False

fig, ejes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

datos = [("Verano", verano, resultados_verano, ejes[0], v_esperada_verano),
         ("Invierno", invierno, resultados_invierno, ejes[1], v_esperada_invierno)]

for nombre, espectro, resultados, eje, v_epoca in datos:
    region = espectro[(espectro["lambda"] > lambda_min) & (espectro["lambda"] < lambda_max)]
    lineas_region = catalogo_A[(catalogo_A["Lvac"] > lambda_min - 2) & (catalogo_A["Lvac"] < lambda_max + 2)]
    detectadas = resultados[(resultados["lambda_min"] > lambda_min) &
                            (resultados["lambda_min"] < lambda_max)]

    eje.plot(region["lambda"], region["intensity"], color="black", label="Espectro")
    posiciones = lineas_region["Lvac"]
    if desplazar_catalogo:
        posiciones = posiciones * (1 + v_epoca / c)
    eje.vlines(posiciones, ymin=0, ymax=1.05, color="red", alpha=0.5,
               label="Líneas de Nave (calidad A)")
    eje.scatter(detectadas["lambda_min"], detectadas["flujo_centro"], color="green",
                s=25, zorder=3, label="Detectadas")
    eje.set_ylabel("Flujo normalizado")
    eje.set_title(f"{nombre}: {lambda_min} - {lambda_max} Å")
    eje.legend(loc="lower right")

ejes[1].set_xlabel("Longitud de onda (Å)")
plt.tight_layout()
plt.show()

In [ ]:
# Usa el mismo lambda_min y lambda_max de la celda anterior (conviene una región chica)

for nombre, espectro, resultados in [("Verano", verano, resultados_verano),
                                     ("Invierno", invierno, resultados_invierno)]:
    lam = espectro["lambda"].values
    flujo = espectro["intensity"].values
    seleccion = resultados[(resultados["lambda_min"] > lambda_min) &
                           (resultados["lambda_min"] < lambda_max)]
    if len(seleccion) == 0:
        print(nombre, ": no hay líneas detectadas en esta región")
        continue

    columnas = 4
    filas = int(np.ceil(len(seleccion) / columnas))
    fig, ejes = plt.subplots(filas, columnas, figsize=(16, 3.2 * filas), squeeze=False)
    ejes = ejes.flatten()

    for eje, (_, linea) in zip(ejes, seleccion.iterrows()):
        i_min = int(linea["i_min"])
        i_izq = int(linea["i_izq"])
        i_der = int(linea["i_der"])

        # Ala completa en gris, puntos del ajuste en negro
        eje.plot(lam[i_izq:i_der + 1], flujo[i_izq:i_der + 1], ".", color="grey", markersize=4)
        ajuste = ajustar_cabecita(lam, flujo, i_min, linea["profundidad"])
        eje.plot(ajuste["x"], ajuste["y"], "o", color="black", markersize=5)

        # Polinomio
        x_curva = np.linspace(ajuste["x"][0], ajuste["x"][-1], 100)
        eje.plot(x_curva, ajuste["poli"](x_curva - ajuste["x0"]), color="blue")

        # Mínimo del polinomio, línea del catálogo y bisectriz
        eje.plot(linea["lambda_min"], linea["flujo_centro"], "o", color="green", markersize=7)
        eje.axvline(linea["lambda_catalogo"], color="red", alpha=0.6)
        bis_flujo, bis_lambda = calcular_bisectriz(lam, flujo, i_izq, i_min, i_der)
        eje.plot(bis_lambda, bis_flujo, color="orange")

        eje.set_title(f"{linea['lambda_catalogo']:.2f} Å  ({int(linea['numero_puntos'])} puntos)",
                      fontsize=9)

    for eje in ejes[len(seleccion):]:
        eje.axis("off")

    fig.suptitle(f"{nombre}: negro = puntos del ajuste, azul = polinomio, verde = mínimo, "
                 f"naranja = bisectriz, rojo = catálogo")
    plt.tight_layout()
    plt.show()

In [ ]:
# Fila 1 = verano, fila 2 = invierno
# Columna 1: color = asimetría (azul: ala azul más extendida; rojo: ala roja más extendida)
# Columna 2: color = curvatura

fig, ejes = plt.subplots(2, 2, figsize=(13, 9))

for fila, (nombre, resultados) in enumerate([("Verano", resultados_verano),
                                             ("Invierno", resultados_invierno)]):
    # Para que unos pocos puntos extremos no aplasten la escala de colores
    limite = np.percentile(np.abs(resultados["asimetria"]), 95)

    puntos = ejes[fila, 0].scatter(resultados["velocidad"], resultados["flujo_centro"],
                                   c=resultados["asimetria"], cmap="coolwarm_r",
                                   vmin=-limite, vmax=limite, s=18)
    plt.colorbar(puntos, ax=ejes[fila, 0], label="Asimetría (f''')")
    ejes[fila, 0].set_title(f"{nombre}: color = asimetría")

    limite_curvatura = np.percentile(resultados["curvatura"], 95)
    puntos = ejes[fila, 1].scatter(resultados["velocidad"], resultados["flujo_centro"],
                                   c=resultados["curvatura"], cmap="viridis",
                                   vmin=0, vmax=limite_curvatura, s=18)
    plt.colorbar(puntos, ax=ejes[fila, 1], label="Curvatura (f'' · λ²)")
    ejes[fila, 1].set_title(f"{nombre}: color = curvatura")

    for columna in range(2):
        ejes[fila, columna].set_xlabel("Velocidad Doppler (km/s)")
        ejes[fila, columna].set_ylabel("Flujo en el centro")

plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
rojo_a_negro = LinearSegmentedColormap.from_list("rojo_a_negro", ["red", "black"])

fig, ejes = plt.subplots(1, 2, figsize=(13, 4.5))

for eje, (nombre, resultados) in zip(ejes, [("Verano", resultados_verano),
                                            ("Invierno", resultados_invierno)]):
    puntos = eje.scatter(resultados["flujo_centro"], resultados["curvatura"],
                         c=resultados["lambda_min"], cmap=rojo_a_negro, s=18)
    plt.colorbar(puntos, ax=eje, label="Longitud de onda (Å)")
    eje.set_xlabel("Flujo en el centro")
    eje.set_ylabel("Curvatura (f'' · λ²)")
    eje.set_title(nombre)

plt.tight_layout()
plt.show()

In [ ]:
fig, ejes = plt.subplots(2, 2, figsize=(13, 9))

for fila, (nombre, resultados) in enumerate([("Verano", resultados_verano),
                                             ("Invierno", resultados_invierno)]):
    numero_de_onda = 1e8 / resultados["lambda_min"]        # cm^-1

    puntos = ejes[fila, 0].scatter(resultados["lambda_min"], resultados["potencial_excitacion"],
                                   c=resultados["flujo_centro"], cmap="viridis", s=18)
    plt.colorbar(puntos, ax=ejes[fila, 0], label="Flujo en el centro")
    ejes[fila, 0].set_xlabel("Longitud de onda (Å)")

    puntos = ejes[fila, 1].scatter(numero_de_onda, resultados["potencial_excitacion"],
                                   c=resultados["flujo_centro"], cmap="viridis", s=18)
    plt.colorbar(puntos, ax=ejes[fila, 1], label="Flujo en el centro")
    ejes[fila, 1].set_xlabel("Número de onda (cm$^{-1}$)")

    for columna in range(2):
        ejes[fila, columna].set_ylabel("Potencial de excitación (eV)")
        ejes[fila, columna].set_title(nombre)

plt.tight_layout()
plt.show()

In [ ]:
columnas_excel = [
    "lambda_catalogo", "lambda_min", "velocidad", "flujo_centro", "profundidad",
    "curvatura", "asimetria", "ancho_km_s", "cociente_alas", "rugosidad_bisectriz",
    "diferencia_mitades", "meseta_km_s", "potencial_excitacion", "numero_puntos",
]

parametros = pd.DataFrame({
    "parametro": ["profundidad_minima", "tolerancia (km/s)", "separacion_blend (km/s)",
                  "ancho_ala (km/s)", "tope_minimo", "ala_minima (km/s)", "ancho_minimo (km/s)",
                  "ancho_maximo (km/s)", "cociente_alas_min", "cociente_alas_max",
                  "curvatura_relativa_max", "rugosidad_maxima (km/s)", "diferencia_mitades_max",
                  "meseta_maxima (km/s)", "meseta_margen", "meseta_pendiente", "v_esperada_verano (km/s)", "v_esperada_invierno (km/s)"],
    "valor": [profundidad_minima, tolerancia, separacion_blend, ancho_ala, tope_minimo,
              ala_minima, ancho_minimo, ancho_maximo, cociente_alas_min, cociente_alas_max,
              curvatura_relativa_max, rugosidad_maxima, diferencia_mitades_max,
              meseta_maxima, meseta_margen, meseta_pendiente, v_esperada_verano, v_esperada_invierno],
})

parametros.loc[len(parametros)] = ["k_atipico (comparación)", k_atipico]
parametros.loc[len(parametros)] = ["nivel_combinacion", nivel_combinacion]
parametros.loc[len(parametros)] = ["grado_tendencia", grado_tendencia]

with pd.ExcelWriter("lineas_arturo.xlsx", engine="openpyxl") as escritor:
    resultados_verano[columnas_excel].round(6).to_excel(escritor, sheet_name="Verano", index=False)
    resultados_invierno[columnas_excel].round(6).to_excel(escritor, sheet_name="Invierno", index=False)
    parametros.to_excel(escritor, sheet_name="Parametros", index=False)

    # Atípicas con el motivo (solo si corriste la comparación de atípicas)
    columnas_atipicas = ["lambda_catalogo", "lambda_min", "profundidad", "curv_adim", "asim_adim",
                         "ancho_km_s", "cociente_alas", "rugosidad_bisectriz", "diferencia_mitades",
                         "meseta_km_s", "puntaje", "motivos"]
    for nombre_hoja, t in [("Atipicas_Verano", comparadas_verano), ("Atipicas_Invierno", comparadas_invierno)]:
        t[t["atipica"]].sort_values("puntaje", ascending=False)[columnas_atipicas].round(4).to_excel(
            escritor, sheet_name=nombre_hoja, index=False)

    # columnas más anchas para que se lean los títulos
    for hoja in escritor.sheets.values():
        for columna in hoja.columns:
            hoja.column_dimensions[columna[0].column_letter].width = 24

print("Guardado: lineas_arturo.xlsx")

# SEGUNDO ANÁLISIS: cero por bloque + selecciones alternativas

**Qué cambió respecto a la parte de arriba (que se deja tal cual):**
* La selección *principal* ahora es **solo MAD** (`usar_mahalanobis = False` en la celda de líneas limpias).
* Aquí se comparan tres selecciones: *solo MAD* (principal), *MAD + Mahalanobis* (la anterior) y
  *sin circularidad* (solo rechaza por ancho, rugosidad de bisectriz y mesetas, o sea por **mezcla**,
  sin mirar curvatura ni asimetría, que son justo lo que se quiere estudiar).
* Se corrige el cero de velocidad por bloque (hallazgo del `readme.dat` de Hinkle: cada página tiene su
  propia corrección Doppler; el "fantasma" de ~11 km/s son páginas con otro cero).

**Por qué la selección "sin circularidad":** si se rechazan líneas por tener curvatura o asimetría
extremas y luego se estudia la curvatura o la asimetría, la distribución queda recortada por construcción.
Los valores extremos no se quitan: se *muestran* con una escala de color que no se deja arrastrar por ellos.

In [ ]:
# ============================================================
# 1) Las tres selecciones
# ============================================================
# Solo MAD: se deja pasar lo que solo rechazaba Mahalanobis ("atipica_combinacion").
def solo_mad(tabla):
    return tabla[~tabla["atipica"] | tabla["atipica_combinacion"]].reset_index(drop=True)

def con_mahalanobis(tabla):
    return tabla[~tabla["atipica"]].reset_index(drop=True)

# Sin circularidad: rechazar SOLO por métricas de mezcla/ancho (no por curvatura ni asimetría).
# Usa las columnas z_* que ya calculó la comparación de atípicas (mismo k_atipico).
metricas_de_mezcla = ["ancho_km_s", "rugosidad_bisectriz", "meseta_km_s"]

def sin_circularidad(tabla):
    marca = np.zeros(len(tabla), dtype=bool)
    for col in metricas_de_mezcla:
        lados = vigiladas[col][1]                  # "alto" o "ambos", como en la comparación original
        z = tabla["z_" + col].values
        marca |= (z > k_atipico) if lados == "alto" else (np.abs(z) > k_atipico)
    return tabla[~marca].reset_index(drop=True)

selecciones = {
    "solo MAD (principal)":  (solo_mad(comparadas_verano),         solo_mad(comparadas_invierno)),
    "MAD + Mahalanobis":     (con_mahalanobis(comparadas_verano),  con_mahalanobis(comparadas_invierno)),
    "sin circularidad":      (sin_circularidad(comparadas_verano), sin_circularidad(comparadas_invierno)),
}
print(f"{'selección':26s} {'verano':>7s} {'invierno':>9s}")
print(f"{'candidatas':26s} {len(comparadas_verano):7d} {len(comparadas_invierno):9d}")
for nombre, (v, i) in selecciones.items():
    print(f"{nombre:26s} {len(v):7d} {len(i):9d}")

In [ ]:
# ============================================================
# 2) Bloques de velocidad y cero por bloque
# ============================================================
def mad_robusto(x):
    # MAD = mediana de |x - mediana(x)|; con 1.4826 equivale a una desviación estándar si los datos fueran
    # gaussianos, pero un valor extremo no lo mueve.
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    return 1.4826 * np.median(np.abs(x - np.median(x)))


def detectar_bloques(lim_v, lim_i, salto_minimo=1.5):
    # dv = velocidad verano - velocidad invierno de la MISMA línea. No depende de la física de la línea,
    # solo de las fechas de observación de su página. Donde dv salta, cambia el bloque.
    pares = pd.merge(lim_v[["lambda_catalogo", "lambda_min", "velocidad"]],
                     lim_i[["lambda_catalogo", "lambda_min", "velocidad"]],
                     on="lambda_catalogo", suffixes=("_v", "_i"))
    pares["dv"] = pares["velocidad_v"] - pares["velocidad_i"]
    pares = pares.sort_values("lambda_min_v").reset_index(drop=True)
    dv_suave = pares["dv"].rolling(3, center=True, min_periods=1).median().values
    lam = pares["lambda_min_v"].values
    return [0.5 * (lam[k - 1] + lam[k]) for k in range(1, len(pares))
            if abs(dv_suave[k] - dv_suave[k - 1]) > salto_minimo]


salto_minimo = 1.5
bordes = detectar_bloques(*selecciones["solo MAD (principal)"], salto_minimo)
# Para fijar los bordes a mano (p. ej. con fechas de la monografía): bordes = [11356, 14226]
print("Bordes de bloque detectados (Å):", [round(b) for b in bordes])


def corregir_por_bloque(lim, bordes):
    t = lim.copy()
    t["bloque"] = np.searchsorted(bordes, t["lambda_min"].values)
    t["cero_bloque"] = t.groupby("bloque")["velocidad"].transform("median")
    t["velocidad_rel"] = t["velocidad"] - t["cero_bloque"]
    return t


corregidas = {nombre: (corregir_por_bloque(v, bordes), corregir_por_bloque(i, bordes))
              for nombre, (v, i) in selecciones.items()}
v_main, i_main = corregidas["solo MAD (principal)"]

resumen = []
for epoca, t in [("Verano", v_main), ("Invierno", i_main)]:
    for b, g in t.groupby("bloque"):
        resumen.append({"época": epoca, "bloque": b, "λ desde (Å)": round(g["lambda_min"].min()),
                        "λ hasta (Å)": round(g["lambda_min"].max()), "N líneas": len(g),
                        "cero del bloque (km/s)": round(g["cero_bloque"].iloc[0], 2),
                        "dispersión MAD (km/s)": round(mad_robusto(g["velocidad_rel"]), 2)})
display(pd.DataFrame(resumen))
if min(r["N líneas"] for r in resumen) < 10:
    print("AVISO: hay bloques con menos de 10 líneas; su 'cero' es poco fiable.")

In [ ]:
# ============================================================
# 3) Figura: antes y después de corregir el cero por bloque
# ============================================================
colores_bloque = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple"]
fig, ejes = plt.subplots(2, 2, figsize=(13, 9))
for fila, (epoca, t) in enumerate([("Verano", v_main), ("Invierno", i_main)]):
    for b, g in t.groupby("bloque"):
        etiqueta = f"bloque {b}: {g['lambda_min'].min():.0f}–{g['lambda_min'].max():.0f} Å"
        ejes[fila, 0].scatter(g["velocidad"], g["flujo_centro"], s=16, color=colores_bloque[b], label=etiqueta)
        ejes[fila, 1].scatter(g["velocidad_rel"], g["flujo_centro"], s=16, color=colores_bloque[b])
    ejes[fila, 0].set_title(f"{epoca}: velocidad observada (con el 'fantasma')")
    ejes[fila, 1].set_title(f"{epoca}: velocidad relativa al cero de su bloque")
    ejes[fila, 0].set_xlabel("Velocidad Doppler (km/s)")
    ejes[fila, 1].set_xlabel("Velocidad relativa (km/s)")
    ejes[fila, 1].set_xlim(-3, 3)
    ejes[fila, 0].legend(fontsize=7)
    for col in range(2):
        ejes[fila, col].set_ylabel("Flujo en el centro")
plt.tight_layout()
plt.show()

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# la misma escala de colores para verano e invierno
lambda_menor = min(resultados_verano["lambda_catalogo"].min(), resultados_invierno["lambda_catalogo"].min())
lambda_mayor = max(resultados_verano["lambda_catalogo"].max(), resultados_invierno["lambda_catalogo"].max())
escala = plt.Normalize(vmin=lambda_menor, vmax=lambda_mayor)
arcoiris = plt.cm.rainbow

for eje, (nombre, espectro, resultados) in zip(ejes, [("Verano", verano, resultados_verano),
                                                      ("Invierno", invierno, resultados_invierno)]):
    lam = espectro["lambda"].values
    flujo = espectro["intensity"].values

    for _, linea in resultados.iterrows():
        bis_flujo, bis_lambda = calcular_bisectriz(lam, flujo, int(linea["i_izq"]),
                                                   int(linea["i_min"]), int(linea["i_der"]))
        # velocidad de cada punto de la bisectriz respecto a la longitud de onda natural
        bis_velocidad = c * (bis_lambda - linea["lambda_catalogo"]) / linea["lambda_catalogo"]
        color = arcoiris(escala(linea["lambda_catalogo"]))
        eje.plot(bis_velocidad, bis_flujo, color=color, alpha=0.7, linewidth=1)

    eje.set_xlabel("Velocidad de la bisectriz (km/s)")
    eje.set_title(nombre)

ejes[0].set_ylabel("Flujo normalizado")
barra = plt.cm.ScalarMappable(norm=escala, cmap=arcoiris)
fig.colorbar(barra, ax=ejes, label="Longitud de onda (Å)")
plt.show()

## Exportar ventanas y doblete para Overleaf
Estas celdas utilizan los resultados de la ejecución actual. Ejecuta primero las celdas de detección, ajuste y filtro MAD.


In [ ]:
# ============================================================
# EXPORTAR VENTANAS DE DETECCION Y DETALLE DEL DOBLETE
# Pegar DESPUES de la celda que define resultados_verano/invierno.
# Conserva el umbral y la seleccion de la ejecucion actual.
# ============================================================
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

RUTA_GRAFICAS = Path('graficas_tesis')
RUTA_DETECCION = RUTA_GRAFICAS / 'deteccion_k327'
RUTA_DETALLE = RUTA_GRAFICAS / 'detalle'
RUTA_DETECCION.mkdir(parents=True, exist_ok=True)
RUTA_DETALLE.mkdir(parents=True, exist_ok=True)

VENTANAS_DETECCION = [(10050, 10100), (10100, 10200),
                      (10200, 10300), (13100, 13200)]


def dibujar_ventana_en_eje(ax, espectro, seleccion, lam_inicial, lam_final,
                           epoca, mostrar_leyenda=False):
    region = espectro.loc[espectro['lambda'].between(lam_inicial, lam_final)]
    lineas = seleccion.loc[seleccion['lambda_min'].between(lam_inicial, lam_final)]
    atlas = catalogo_A.loc[catalogo_A['Lvac'].between(lam_inicial, lam_final)]
    ax.plot(region['lambda'], region['intensity'], color='black', linewidth=1.0)
    for pos in atlas['Lvac']:
        ax.axvline(pos, color='#d62728', lw=0.8, alpha=0.52)
    ax.scatter(lineas['lambda_min'], lineas['flujo_centro'],
               color='#22a35e', edgecolor='black', linewidth=0.3,
               s=37, zorder=4)
    ax.set_xlim(lam_inicial, lam_final)
    ax.set_ylim(0, 1.09)
    ax.set_ylabel('Flujo normalizado')
    ax.set_title(f'{epoca} · {len(lineas)} líneas aceptadas', fontsize=11, loc='left')
    ax.grid(alpha=0.12)
    if mostrar_leyenda:
        handles = [Line2D([0],[0], color='black', lw=1.3, label='Espectro'),
                   Line2D([0],[0], color='#d62728', lw=1.1, label='Nave A (reposo)'),
                   Line2D([0],[0], marker='o', lw=0, color='#22a35e',
                          markersize=6, label='Aceptadas')]
        ax.legend(handles=handles, fontsize=8, loc='lower right', ncol=3)


for li, lf in VENTANAS_DETECCION:
    fig, ejes = plt.subplots(2, 1, figsize=(12.0, 6.0), sharex=True)
    dibujar_ventana_en_eje(ejes[0], verano, resultados_verano, li, lf, 'Verano', True)
    dibujar_ventana_en_eje(ejes[1], invierno, resultados_invierno, li, lf, 'Invierno')
    ejes[-1].set_xlabel('Longitud de onda (Å)')
    fig.suptitle(f'Detección Fe I · {li}–{lf} Å · MAD k={k_atipico:g}', fontsize=13)
    fig.tight_layout()
    destino = RUTA_DETECCION / f'deteccion_{li}_{lf}_k327.png'
    fig.savefig(destino, dpi=190, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print('Guardada:', destino)

# Un detalle adicional: la región de verano con el doblete.
fig, ejes = plt.subplots(2, 1, figsize=(12, 6.3),
                         gridspec_kw={'height_ratios':[2,1.15]})
dibujar_ventana_en_eje(ejes[0], verano, resultados_verano,
                       10420, 10520, 'Verano: región completa', True)
dibujar_ventana_en_eje(ejes[1], verano, resultados_verano,
                       10424.8, 10428.5, 'Ampliación del doblete')
ejes[1].set_xlabel('Longitud de onda (Å)')
ejes[1].ticklabel_format(style='plain', useOffset=False, axis='x')
fig.suptitle('Dos transiciones Fe I cercanas a 10426 Å · verano', fontsize=13)
fig.tight_layout()
destino = RUTA_DETALLE / 'doblete_verano_10420_10520_k327.png'
fig.savefig(destino, dpi=190, bbox_inches='tight')
plt.show()
plt.close(fig)
print('Guardada:', destino)


## Exportar bisectrices individuales por región
Estas celdas utilizan los resultados de la ejecución actual. Ejecuta primero las celdas de detección, ajuste y filtro MAD.


In [ ]:
# ============================================================
# EXPORTAR BISECTRICES INDIVIDUALES POR REGION
# Pegar DESPUES de las celdas de ajuste y de seleccion MAD.
# Cambia REGION_BIS, EPOCA_BIS y MAX_LINEAS_BIS a voluntad.
# ============================================================
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

REGION_BIS = (10050, 10300)
EPOCA_BIS = 'verano'  # 'verano' o 'invierno'
MAX_LINEAS_BIS = 4
MARGEN_PERFIL_KM_S = 18

CARPETA_BIS = Path('graficas_tesis') / 'bisectrices_k327'
CARPETA_BIS.mkdir(parents=True, exist_ok=True)

if EPOCA_BIS.lower() == 'verano':
    esp_bis, tabla_bis, titulo_bis = verano, resultados_verano, 'Verano'
else:
    esp_bis, tabla_bis, titulo_bis = invierno, resultados_invierno, 'Invierno'

candidatas_bis = tabla_bis.loc[
    tabla_bis['lambda_min'].between(*REGION_BIS)
].copy().sort_values('lambda_min')

# Selección reproducible: repartimos la región en tantos tramos como
# ejemplos solicitados y elegimos, en cada uno, la menor rugosidad.
# Esto evita escoger varios perfiles juntos y reduce bisectrices con picos.
if len(candidatas_bis) > MAX_LINEAS_BIS:
    limites = np.linspace(REGION_BIS[0], REGION_BIS[1], MAX_LINEAS_BIS+1)
    escogidas = []
    for j in range(MAX_LINEAS_BIS):
        bloque = candidatas_bis.loc[
            (candidatas_bis['lambda_min'] >= limites[j]) &
            (candidatas_bis['lambda_min'] <= limites[j+1])
        ]
        if not bloque.empty:
            escogidas.append(bloque.sort_values('rugosidad_bisectriz').index[0])
    if len(escogidas) < MAX_LINEAS_BIS:
        restantes = candidatas_bis.drop(index=escogidas).sort_values('rugosidad_bisectriz')
        escogidas += list(restantes.head(MAX_LINEAS_BIS-len(escogidas)).index)
    candidatas_bis = candidatas_bis.loc[escogidas].sort_values('lambda_min')


def graficar_bisectriz_en_ejes(ax_perfil, ax_bis, fila, espectro, rotulo):
    lam = espectro['lambda'].to_numpy()
    flujo = espectro['intensity'].to_numpy()
    i_min, i_izq, i_der = (int(fila[col]) for col in ['i_min','i_izq','i_der'])
    ajuste = ajustar_cabecita(lam, flujo, i_min, fila['profundidad'])
    if ajuste is None:
        ax_perfil.text(.5,.5,'Ajuste no disponible',ha='center',transform=ax_perfil.transAxes)
        return
    f_b, lam_b = calcular_bisectriz(lam, flujo, i_izq, i_min, i_der)
    lmin = float(ajuste['lambda_min'])
    u = c*(lam-lmin)/lmin
    v_b = c*(lam_b-lmin)/lmin
    sel = (u>=-MARGEN_PERFIL_KM_S) & (u<=MARGEN_PERFIL_KM_S)

    xx = np.linspace(ajuste['x'].min(),ajuste['x'].max(),400)
    pp = ajuste['poli'](xx-ajuste['x0'])
    vxx = c*(xx-lmin)/lmin
    ptsv = c*(ajuste['x']-lmin)/lmin
    ax_perfil.plot(u[sel], flujo[sel], color='black', lw=1.2, label='Espectro')
    ax_perfil.plot(vxx,pp,color='#3566bb',lw=1.5,label='Polinomio grado 4')
    ax_perfil.scatter(ptsv,ajuste['y'],color='black',edgecolor='white',s=26,zorder=4,label='7 muestras')
    ax_perfil.plot(v_b,f_b,color='#19995c',lw=1.7,label='Bisectriz')
    vl0=c*(float(fila['lambda_catalogo'])-lmin)/lmin
    if -MARGEN_PERFIL_KM_S < vl0 < MARGEN_PERFIL_KM_S:
        ax_perfil.axvline(vl0,ls='--',lw=0.9,color='#d62728',label='Nave (reposo)')
    ax_perfil.axvline(0,ls=':',lw=1,color='#666666')
    ax_perfil.set_xlim(-MARGEN_PERFIL_KM_S,MARGEN_PERFIL_KM_S)
    ax_perfil.set_xlabel('Velocidad respecto al mínimo (km/s)')
    ax_perfil.set_ylabel('Flujo')
    ax_perfil.set_title(f'{rotulo} · λ lab.={fila["lambda_catalogo"]:.3f} Å',fontsize=11,loc='left')
    ax_perfil.grid(alpha=.12)

    # En un segundo eje, la bisectriz se amplía horizontalmente. Así se
    # aprecia su curvatura, que sería invisible con la escala del perfil.
    ax_bis.plot(v_b, f_b, color='#19995c', lw=1.7)
    ax_bis.scatter(v_b[::8],f_b[::8],color='#19995c',s=10)
    pad=max(0.075, 0.20*np.ptp(v_b))
    ax_bis.set_xlim(np.min(v_b)-pad, np.max(v_b)+pad)
    ax_bis.set_ylim(np.min(f_b)-0.025, np.max(f_b)+0.025)
    ax_bis.set_xlabel('Velocidad de la bisectriz (km/s)')
    ax_bis.set_ylabel('Flujo')
    ax_bis.set_title('Bisectriz ampliada',fontsize=11,loc='left')
    ax_bis.grid(alpha=.18)


if candidatas_bis.empty:
    print('No hay líneas aceptadas en la región solicitada.')
else:
    fig, ejes = plt.subplots(len(candidatas_bis),2,
                            figsize=(12,3.45*len(candidatas_bis)),squeeze=False)
    for n,(_,fila) in enumerate(candidatas_bis.iterrows()):
        graficar_bisectriz_en_ejes(ejes[n,0],ejes[n,1],fila,esp_bis,titulo_bis)
        # Cada línea se guarda también como figura individual.
        f_ind, ax_ind = plt.subplots(1,2,figsize=(11.5,3.7))
        graficar_bisectriz_en_ejes(ax_ind[0],ax_ind[1],fila,esp_bis,titulo_bis)
        f_ind.tight_layout()
        ruta = CARPETA_BIS / f'bisectriz_{EPOCA_BIS.lower()}_{fila["lambda_catalogo"]:.3f}_k327.png'
        f_ind.savefig(ruta,dpi=190,bbox_inches='tight')
        plt.close(f_ind)
    fig.suptitle(f'Bisectrices de Fe I · {titulo_bis} · {REGION_BIS[0]}–{REGION_BIS[1]} Å',
                 fontsize=14,y=1.005)
    fig.tight_layout()
    ruta = CARPETA_BIS / f'bisectrices_{EPOCA_BIS.lower()}_{REGION_BIS[0]}_{REGION_BIS[1]}_k327.png'
    fig.savefig(ruta,dpi=190,bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f'Guardadas {len(candidatas_bis)} bisectrices individuales y galería: {ruta}')


## Figuras adicionales de la versión final (k = 3.27)

Las siguientes celdas son **complementarias**: no alteran el método de detección, los ajustes de 7 puntos, el umbral MAD ni las tablas `resultados_verano`/`resultados_invierno`. Se agregan aquí para reunir en un único notebook las dos tandas de curvatura que se mostraron en la última versión de la tesis y las bisectrices globales en tres colores.

**Ejecución:** realiza todas las celdas anteriores en orden y después ejecuta las siguientes. Los archivos se guardan en `graficas_tesis/`. La variable `k_atipico` debe valer `3.27`.

In [ ]:
# ============================================================
# FIGURAS DE CURVATURA: SIN AJUSTE Y CON CURVA CUBICA
# Mismas candidatas y seleccion final de la ejecucion actual.
# No se modifican las tablas ni el algoritmo de seleccion.
# ============================================================
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patheffects as pe
from scipy.stats import spearmanr

if abs(float(k_atipico) - 3.27) > 1e-12:
    print(f'ATENCION: k_atipico = {k_atipico}; estas figuras no corresponden a k=3.27')

CARPETA_CURV = Path('graficas_tesis') / 'propiedades'
CARPETA_CURV.mkdir(parents=True, exist_ok=True)
TABLAS_CURV = Path('tablas')
TABLAS_CURV.mkdir(exist_ok=True)

MUESTRAS_CURV = {
    'perm_v': comparadas_verano,
    'perm_i': comparadas_invierno,
    'fin_v': resultados_verano,
    'fin_i': resultados_invierno,
}
MAPA_CURV = mpl.colors.LinearSegmentedColormap.from_list(
    'rojo_negro_artu', ['#f32932', '#381b22', '#050505'])
NORM_CURV = mpl.colors.Normalize(vmin=9000, vmax=50000)

AJUSTES_CURV = {}
for clave, tabla in MUESTRAS_CURV.items():
    x = tabla['flujo_centro'].to_numpy(dtype=float)
    y = tabla['curvatura'].to_numpy(dtype=float) / 1e9
    matriz = np.column_stack([x, x**2, x**3])
    abc = np.linalg.lstsq(matriz, y, rcond=None)[0]
    estimada = matriz @ abc
    r2 = 1 - np.sum((y-estimada)**2) / np.sum((y-y.mean())**2)
    AJUSTES_CURV[clave] = {'coef': abc, 'R2': float(r2),
                           'rho_Spearman': float(spearmanr(x, y).statistic),
                           'N': len(tabla)}

pd.DataFrame([
    {'muestra': clave, 'N': d['N'], 'a': d['coef'][0],
     'b': d['coef'][1], 'c': d['coef'][2], 'R2': d['R2'],
     'rho_Spearman': d['rho_Spearman']}
    for clave, d in AJUSTES_CURV.items()
]).to_csv(TABLAS_CURV/'ajustes_cubicos_curvatura_k327.csv', index=False)

for con_ajuste in (False, True):
    fig = plt.figure(figsize=(12, 8.1), facecolor='white')
    gs = fig.add_gridspec(2, 3, left=.083, right=.932, top=.935,
                          bottom=.112, width_ratios=[1, 1, .035],
                          wspace=.235, hspace=.38)
    ejes = [[fig.add_subplot(gs[fila, col]) for col in range(2)]
            for fila in range(2)]
    eje_barra = fig.add_subplot(gs[:, 2])
    ultimo_scatter = None
    for fila, prefijo in enumerate(('perm', 'fin')):
        tablas_fila = [MUESTRAS_CURV[f'{prefijo}_v'],
                       MUESTRAS_CURV[f'{prefijo}_i']]
        ymax = max(t['curvatura'].max() for t in tablas_fila) / 1e9
        ymin = min(t['curvatura'].min() for t in tablas_fila) / 1e9
        ylim = ((-max(.5, -ymin+.3), max(1.3, 1.07*ymax))
                if fila == 0 else (.40, 1.13*ymax))
        for col, epoca in enumerate(('v', 'i')):
            clave = f'{prefijo}_{epoca}'
            tabla = MUESTRAS_CURV[clave]
            x = tabla['flujo_centro'].to_numpy(dtype=float)
            y = tabla['curvatura'].to_numpy(dtype=float) / 1e9
            ax = ejes[fila][col]
            ultimo_scatter = ax.scatter(
                x, y, c=tabla['lambda_catalogo'], cmap=MAPA_CURV,
                norm=NORM_CURV, s=20 if fila == 0 else 25,
                linewidths=0, alpha=.90, zorder=2)
            if con_ajuste:
                abc = AJUSTES_CURV[clave]['coef']
                f = lambda xx: abc[0]*xx + abc[1]*xx**2 + abc[2]*xx**3
                xs = np.linspace(x.min(), x.max(), 240)
                ax.plot(xs, f(xs), color='black', lw=2.05, zorder=5,
                        path_effects=[pe.Stroke(linewidth=3.6, foreground='white'), pe.Normal()])
                xs0 = np.linspace(0, x.min(), 90)
                ax.plot(xs0, f(xs0), color='#444444', ls='--', lw=1.2, alpha=.65)
                ax.text(.98, .96, f'$R^2={AJUSTES_CURV[clave]["R2"]:.3f}$',
                        ha='right', va='top', transform=ax.transAxes,
                        fontsize=9.3, bbox=dict(facecolor='white', edgecolor='none', alpha=.88))
            ax.set_title(f'{"Verano" if col == 0 else "Invierno"} · {len(tabla)} líneas', fontsize=11)
            ax.set_xlim(-.02, .97)
            ax.set_ylim(*ylim)
            ax.set_xlabel('Flujo central')
            if col == 0:
                ax.set_ylabel(r'Curvatura $\lambda^2 P\prime\prime$ ($10^9$)')
            else:
                ax.tick_params(labelleft=False)
            ax.grid(alpha=.15, lw=.65)
            ax.set_axisbelow(True)
    barra = fig.colorbar(ultimo_scatter, cax=eje_barra)
    barra.set_label('Longitud de onda de laboratorio (Å)')
    archivo = ('curvatura_4paneles_con_cubica.png' if con_ajuste
               else 'curvatura_4paneles_sin_ajuste.png')
    fig.savefig(CARPETA_CURV/archivo, dpi=245, facecolor='white')
    plt.close(fig)
    print('Guardada:', CARPETA_CURV/archivo)

print('Ajustes calculados para:', {k: (v['N'], round(v['R2'], 3))
                                    for k,v in AJUSTES_CURV.items()})


## Bisectrices globales: tres rangos espectrales

Se dibujan todas las bisectrices calculadas por la función original `calcular_bisectriz`, separadas en tres intervalos de longitud de onda de laboratorio; solo cambia la presentación. Las velocidades se expresan respecto al mínimo refinado de cada línea, no respecto a una velocidad estelar absoluta.

In [ ]:
# ============================================================
# BISECTRICES GLOBALES EN TRES COLORES (k=3.27)
# Figuras complementarias; no alteran resultados.
# ============================================================
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

CARPETA_BIS_GLOBAL = Path('graficas_tesis')/'bisectrices_extra'
CARPETA_BIS_GLOBAL.mkdir(parents=True, exist_ok=True)
RANGOS_LAMBDA = [(9100, 15200), (15200, 16900), (16900, 46500)]
COLORES_RANGOS = ['#c62828', '#2e7d32', '#1565c0']

fig, ejes = plt.subplots(1, 2, figsize=(12.2, 5.6), sharey=True)
for ax, (nombre, espectro, tabla) in zip(ejes, [
    ('Verano', verano, resultados_verano),
    ('Invierno', invierno, resultados_invierno),
]):
    lam = espectro['lambda'].to_numpy()
    flujo = espectro['intensity'].to_numpy()
    for (li, lf), color in zip(RANGOS_LAMBDA, COLORES_RANGOS):
        grupo = tabla[(tabla['lambda_catalogo']>=li) & (tabla['lambda_catalogo']<lf)]
        primer_trazo = True
        for _, linea in grupo.iterrows():
            niveles, lb = calcular_bisectriz(
                lam, flujo, int(linea['i_izq']), int(linea['i_min']), int(linea['i_der']))
            lmin = float(linea['lambda_min'])
            vb = c*(lb-lmin)/lmin
            etiqueta = f'{li}–{lf} Å' if primer_trazo else None
            ax.plot(vb, niveles, color=color, lw=.75, alpha=.42, label=etiqueta)
            primer_trazo = False
    ax.axvline(0, color='0.5', ls='--', lw=.8)
    ax.set_title(f'{nombre}: bisectrices relativas al núcleo')
    ax.set_xlabel('Velocidad de la bisectriz (km/s)')
    ax.grid(alpha=.18)
ejes[0].set_ylabel('Flujo normalizado')
ejes[0].legend(title='Rangos de λ lab.', fontsize=8, title_fontsize=9, loc='lower left')
fig.suptitle('Bisectrices de la muestra final, en tres rangos de longitud de onda',
             y=.98, fontsize=13)
fig.tight_layout(rect=[0, 0, 1, .95])
ruta = CARPETA_BIS_GLOBAL/'bisectrices_peluda_3rangos_k327.png'
fig.savefig(ruta, dpi=220, bbox_inches='tight')
plt.close(fig)
print('Guardada:', ruta)
